In [7]:
import re
from tqdm import tqdm 
from pathlib import Path 
from joblib import Parallel, delayed
import pandas as pd

In [8]:
!which python


/om2/user/imgriff/conda_envs/aligner/bin/python


In [9]:
cv_path = Path('/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/en/')

In [10]:
!ls {cv_path}

clips	 invalidated.tsv  reported.tsv	train.tsv
dev.tsv  other.tsv	  test.tsv	validated.tsv


In [11]:
# get manifest of all valid files:

tsv_path = cv_path / 'validated.tsv'

valid_df = pd.read_csv(tsv_path, sep='\t')
valid_df = valid_df[~pd.isna(valid_df.sentence)]
# valid_df = valid_df[~valid_df.gender.isna()]


In [12]:

# valid_df = valid_df[~valid_df.gender.isna()]

In [15]:
valid_df.head()

,client_id,path,sentence,up_votes,down_votes,age,gender,accents,locale,segment
0,000abb3006b78ea4c1144e55d9d158f05a9db011016051...,common_voice_en_27710027.mp3,"Joe Keaton disapproved of films, and Buster al...",3,1,NaN,NaN,NaN,en,NaN
1,0013037a1d45cc33460806cc3f8ecee9d536c45639ba4c...,common_voice_en_699711.mp3,She'll be all right.,2,1,NaN,NaN,NaN,en,NaN
2,0014c5a3e5715a54855257779b89c2bb498d470b225866...,common_voice_en_21953345.mp3,six,3,2,NaN,NaN,NaN,en,Benchmark
3,001509f4624a7dee75247f6a8b642c4a0d09f8be3eeea6...,common_voice_en_18132047.mp3,All's well that ends well.,2,0,NaN,NaN,NaN,en,NaN
4,001519f234e04528a2b36158c205dbe61c8da45ab0242f...,common_voice_en_27340672.mp3,It is a busy market town that serves a large s...,2,0,NaN,NaN,NaN,en,NaN


In [16]:

# path_names = valid_df['path'].apply(lambda x: x.split('.')[0]).to_list()
# sentences = valid_df['sentence'].str.strip('!.?,').str.replace(',', '').to_list()

In [32]:
from g2p_en import G2p
# g2p model from here: https://github.com/Kyubyong/g2p

import nltk
from nltk.tokenize import WhitespaceTokenizer
import string
import itertools 
import numpy as np
from collections import Counter
import spacy

# nlp = spacy.load('en_core_web_trf')

# nltk.download('words')
# nltk.download('punkt')

en_dict = set(nltk.corpus.words.words())

# en_dict = set(nlp.vocab.strings)


In [4]:
len(en_dict)

83938

In [17]:
tokenizer = WhitespaceTokenizer()


In [18]:
tkn_sentences = valid_df['sentence'].str.strip('!.?,').str.replace(',', '').str.lower().apply(lambda x: tokenizer.tokenize(x))

In [19]:
tkn_sentences = tkn_sentences.to_list()

In [75]:
import re

all_tokens = [re.sub("[^\w\d'\s]+",'', token) for token in itertools.chain.from_iterable(tkn_sentences)]


#

In [76]:
cv_counter = Counter(all_tokens)


In [77]:
len(cv_counter)

280385

In [78]:
cv_counter.total() 
# total unique words 

14667330

In [79]:
cv_words = sorted([str(key).lstrip("'").rstrip("'") for key in cv_counter.keys()])

In [80]:
cv_tokens = Counter(cv_words)
cv_words = list(cv_tokens.keys())

In [81]:
cv_words = cv_words[1:] # remove space at beginning 

In [82]:
len(cv_words)

274358

In [91]:
pattern = "^[A-Za-z_']*$" 

cv_words = [word for word in cv_words if bool(re.match(pattern, word))]

In [92]:
len(cv_words)

273236

In [96]:
cv_path = Path('/scratch2/weka/mcdermott/imgriff/datasets/commonvoice_9/')

with open(cv_path / 'word_list.txt', 'w') as f:
    for word in tqdm(cv_words):
        f.write(f"{word}\n")

100%|██████████| 273236/273236 [00:00<00:00, 1276651.63it/s]


In [97]:
g2p = G2p() # init g2p; on input args accepted 

In [98]:
g2p(cv_words[1])

['EY1', 'B', 'EY0']

## Write pronunciation dict

In [99]:
cv_path = Path('/scratch2/weka/mcdermott/imgriff/datasets/commonvoice_9/')


with open(cv_path / 'new_dictionary.txt', 'w') as f:
    for word in tqdm(cv_words):
        phones = ' '.join(g2p(word)) # join as space-separated string
        f.write(f"{word.upper()}\t{phones}\n")

100%|██████████| 273236/273236 [14:41<00:00, 310.05it/s]


## Save transcripts to directory 

In [11]:
tscript_path = cv_path = Path('/scratch2/weka/mcdermott/imgriff/datasets/commonvoice_9/en/clips/')



In [12]:
valid_df = valid_df.reset_index(drop=True)

In [14]:
valid_df.shape

(1556248, 10)

In [15]:
# eg of commands, run in command line 

for ix in tqdm(range(len(valid_df)), total=len(valid_df)):
    f_name = valid_df['path'][ix]
    sentence = valid_df['sentence'][ix]
    f_name = f"{f_name.split('.')[0]}.txt"
    with open(tscript_path / f_name, 'w') as file:
        file.write(sentence)

  2%|▏         | 26269/1556248 [00:55<53:57, 472.53it/s]  

KeyboardInterrupt

